In [0]:
# replace with your catalog
CATALOG = spark.catalog.currentCatalog()
CATALOG = "nikkthegreek"

In [0]:
import requests
import os
import sys
import platform
from lakehouse.spark import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [0]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

In [0]:
try:
    spark
except NameError:
    builder = (
        SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
        .master("local[4]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

# 1. Set Up and Bronze Data

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

In [0]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [0]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [0]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [0]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {df.count()}")
df.show(truncate=False)

In [0]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [0]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [0]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df


silver_instance = StarWarsSilver(
    spark, catalog=CATALOG, source_schema="bronze", target_schema="silver"
)

In [0]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute(
    "people", "planets"
)

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

# 6 Clean Up

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
#spark.stop()